# Old → New Annotation Conversion Rule

Path B converts old-style structured JSON into the new annotation style before scoring.
This notebook validates that conversion against the real reference files.

**The rule is specified by `data/input/Rules.xlsx`** — a 16-row truth table over whether
each of `title`, `section.title`, `section.description` and `question.text` is empty (E)
or non-empty (N). `dmpbridge.evaluation.annotation_rules` transcribes that sheet directly.

This notebook **imports the shipped function** rather than redefining it. An earlier
revision kept its own copy, which then drifted from the library and validated a rule the
pipeline no longer used.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import copy
import json
import re
from collections import Counter

import openpyxl

from dmpbridge.core import paths
from dmpbridge.evaluation.evaluate import resolve_old_gt_path
from dmpbridge.evaluation.annotation_rules import (
    apply_new_annotation_rules,   # the shipped rule — not a copy
    resolve_new_gt_path,
    _RULES, _state, RULE_FIELDS,
)

print(f"old ground truth: {resolve_old_gt_path(1).parent}")
print(f"new ground truth: {resolve_new_gt_path(1).parent}")

old ground truth: C:\Users\Nahid\dmpbridge\data\input\ground_truth_old_version
new ground truth: C:\Users\Nahid\dmpbridge\data\input\ground_truth_new_version


## Ground-truth filenames

Neither directory follows one consistent naming pattern, and both have been renamed during
the project — the old-version files went from `sampleN_old_dmp.json` to
`sampleN_dmp_old.json`. Both resolvers scan by sample number rather than matching a fixed
pattern, so a future rename will not break them.

In [2]:
print(f"{'sample':<9}{'old version':<26}{'new version'}")
print('-' * 62)
for n in range(1, 11):
    print(f"{n:<9}{resolve_old_gt_path(n).name:<26}{resolve_new_gt_path(n).name}")

sample   old version               new version
--------------------------------------------------------------
1        sample1_dmp_old.json      sample1_dmp_new.json
2        sample2_dmp_old.json      sample2_dmp_new.json
3        sample3_dmp_old.json      sample3_dmp_new.json
4        sample4_dmp_old.json      sample4_dmp_new.json
5        sample5_dmp_old.json      dmp_sample5_new.json
6        sample6_dmp_old.json      dmp_sample6_new.json
7        sample7_dmp_old.json      sample7_dmp_new.json
8        sample8_dmp_old.json      dmp_sample8_new.json
9        sample9_dmp_old.json      dmp_sample9_new.json
10       sample10_dmp_old.json     dmp_sample10_new.json


## The rule table

Read straight from `Rules.xlsx` and compared against the library's transcription, so this
cell fails loudly if the two have drifted apart.

In [3]:
sheet = openpyxl.load_workbook('data/input/Rules.xlsx', data_only=True).worksheets[0]
header = [str(c).strip() if c else '' for c in next(sheet.iter_rows(values_only=True))]

assert header[1:5] == list(RULE_FIELDS), (
    f'Rules.xlsx column order is {header[1:5]}, library expects {list(RULE_FIELDS)}')
print('column order:', ' | '.join(RULE_FIELDS))
print()

pattern_hdr = '/'.join(f.replace('section.', 'sec.').replace('question.text', 'q.text')
                       for f in RULE_FIELDS)
print(f"{'row':>4}  {pattern_hdr:<40}{'action':<44}match")
print('-' * 100)
for row in sheet.iter_rows(min_row=2, max_row=17, values_only=True):
    n, action = row[0], row[5] or ''
    key = tuple(row[1:5])
    m = re.search(r'Copy "?([\w.]+)"? into "?([\w.]+)"?', action)
    expected = (m.group(1), m.group(2)) if m else None
    act = 'leave unchanged' if expected is None else f'{expected[0]} -> {expected[1]}'
    ok = 'OK' if _RULES[key] == expected else 'MISMATCH'
    print(f"{n:>4}  {'/'.join(key):<36}{act:<44}{ok}")

column order: title | section.title | section.description | question.text

 row  title/sec.title/sec.description/q.text  action                                      match
----------------------------------------------------------------------------------------------------
   1  E/E/E/E                             leave unchanged                             OK
   2  E/E/E/N                             leave unchanged                             OK
   3  E/E/N/E                             section.description -> question.text        OK
   4  E/E/N/N                             leave unchanged                             OK
   5  E/N/E/E                             section.title -> question.text              OK
   6  E/N/E/N                             leave unchanged                             OK
   7  E/N/N/E                             section.title -> question.text              OK
   8  E/N/N/N                             leave unchanged                             OK
   9  N/E/E/E   

### What the table says, in one sentence

The sixteen rows express a single rule in two halves:

- **`question.text` already has text** → leave it alone. That is every even row.
- **`question.text` is empty** → fill it from the first available source, in order:

  `section.title` → `section.description` → document title

`question.text` is the only field the rule ever writes. Neither `section.title` nor the
document title is modified.

## Validation against the real reference files

Applies the rule to each old-version file and compares against the corresponding
new-version file. Three comparisons:

- **raw** — equal as JSON, byte for byte
- **content** — equal ignoring `template.version`, which differs only by capitalisation
  (`"V1"` vs `"v1"`) and has nothing to do with annotation
- **rule fields** — equal on just the `(section.title, question.text)` pairs, the only
  thing the rule can affect. This isolates the rule from unrelated text edits.

In [4]:
def without_version(structured: dict) -> dict:
    d = copy.deepcopy(structured)
    d['narrative']['template'].pop('version', None)
    return d


def label_pairs(doc: dict):
    """The (section.title, question.text) pairs the rule can affect."""
    t = doc['narrative']['template']
    return [((s.get('title') or '').strip(), (q.get('text') or '').strip())
            for s in t.get('section', []) for q in s.get('question', [])]


raw_ok = content_ok = pairs_ok = 0
print(f"{'sample':<9}{'raw':>7}{'content':>10}{'rule fields':>14}")
print('-' * 40)
for n in range(1, 11):
    old = json.loads(resolve_old_gt_path(n).read_text(encoding='utf-8'))
    new = json.loads(resolve_new_gt_path(n).read_text(encoding='utf-8'))
    got = apply_new_annotation_rules(old)

    r = got == new
    c = without_version(got) == without_version(new)
    p = label_pairs(got) == label_pairs(new)
    raw_ok += r
    content_ok += c
    pairs_ok += p
    print(f"{n:<9}{str(r):>7}{str(c):>10}{str(p):>14}")
print('-' * 40)
print(f"{'TOTAL':<9}{raw_ok:>5}/10{content_ok:>8}/10{pairs_ok:>12}/10")

sample       raw   content   rule fields
----------------------------------------
1           True      True          True
2          False     False          True
3           True      True          True
4           True      True          True
5          False      True          True
6          False      True          True
7           True      True          True
8          False      True          True
9          False      True          True
10         False      True          True
----------------------------------------
TOTAL        4/10       9/10          10/10


## Which rows actually fire

Only some of the sixteen combinations occur in this corpus. The rest are untested by real
data and are covered only by the unit tests.

In [5]:
ROWNO = {k: i + 1 for i, k in enumerate(_RULES)}
fired, effect = Counter(), Counter()

for n in range(1, 11):
    t = json.loads(resolve_old_gt_path(n).read_text(encoding='utf-8'))['narrative']['template']
    for s in t.get('section', []):
        for q in s.get('question', []):
            vals = {'title': t.get('title'),
                    'section.title': s.get('title'),
                    'section.description': s.get('description'),
                    'question.text': q.get('text')}
            key = tuple(_state(vals[f]) for f in RULE_FIELDS)
            fired[key] += 1
            action = _RULES[key]
            effect['question already had text - left alone' if action is None
                   else f'filled from {action[0]}'] += 1

print(f"{'row':>4}  {pattern_hdr:<40}{'action':<44}{'count':>6}")
print('-' * 100)
for k, v in _RULES.items():
    if not fired.get(k):
        continue
    act = 'leave unchanged' if v is None else f'{v[0]} -> {v[1]}'
    print(f"{ROWNO[k]:>4}  {'/'.join(k):<36}{act:<44}{fired[k]:>6}")

print()
print('rows never exercised by this corpus:',
      [ROWNO[k] for k in _RULES if not fired.get(k)])
print()
for k, v in effect.most_common():
    print(f'  {k:<44}{v:>4}')
print(f"  {'':<44}{'---':>4}")
print(f"  {'total questions':<44}{sum(effect.values()):>4}")

 row  title/sec.title/sec.description/q.text  action                                       count
----------------------------------------------------------------------------------------------------
   9  N/E/E/E                             title -> question.text                           2
  13  N/N/E/E                             section.title -> question.text                  32
  14  N/N/E/N                             leave unchanged                                  9
  15  N/N/N/E                             section.title -> question.text                   6
  16  N/N/N/N                             leave unchanged                                  7

rows never exercised by this corpus: [1, 2, 3, 4, 5, 6, 7, 8, 10, 11, 12]

  filled from section.title                     38
  question already had text - left alone        16
  filled from title                              2
                                               ---
  total questions                               56


## Why the column order matters

On 6 August the spreadsheet's column order changed — `question.text` moved from column 2
to column 4 — and the transcription was not updated with it. The action text was
unchanged, so nothing looked obviously wrong, but six rows became incoherent: they named a
source field the same row marked empty, or wrote into a field the row marked as already
populated. Agreement with the reference files fell from 10/10 to 2/10.

The cell below reproduces that failure directly. Two safeguards now exist: the library
records the expected order in `RULE_FIELDS`, and both this notebook and
`tests/test_annotation_rules.py` assert the sheet's header against it.

In [6]:
WRONG_ORDER = ('title', 'question.text', 'section.title', 'section.description')


def apply_with_field_order(data: dict, order) -> dict:
    """The shipped rule, but keyed with a different column order."""
    data = copy.deepcopy(data)
    template = data.get('narrative', data).get('template', {})
    doc_title = (template.get('title') or '').strip()
    for section in template.get('section', []):
        for question in section.get('question', []):
            vals = {'title': doc_title,
                    'section.title': (section.get('title') or '').strip(),
                    'section.description': (section.get('description') or '').strip(),
                    'question.text': (question.get('text') or '').strip()}
            action = _RULES[tuple(_state(vals[f]) for f in order)]
            if action is None:
                continue
            source, target = action
            if target == 'question.text':
                question['text'] = vals[source]
            elif target == 'section.title':
                section['title'] = vals[source]
    return data


for name, order in (('correct order', RULE_FIELDS), ('wrong order', WRONG_ORDER)):
    ok = sum(
        label_pairs(apply_with_field_order(
            json.loads(resolve_old_gt_path(n).read_text(encoding='utf-8')), order))
        == label_pairs(json.loads(resolve_new_gt_path(n).read_text(encoding='utf-8')))
        for n in range(1, 11)
    )
    print(f"{name:<16}{' | '.join(order)}")
    print(f"{'':<16}reproduces the reference annotation for {ok}/10 samples")
    print()

correct order   title | section.title | section.description | question.text
                reproduces the reference annotation for 10/10 samples

wrong order     title | question.text | section.title | section.description
                reproduces the reference annotation for 2/10 samples



## Convert and save

Applies the rule to every old-version file and writes the result to
`data/output/ground_truth_converted_test/`, so the converted documents can be opened and
read directly.

In [7]:
OUT_DIR = paths.OUTPUT_ROOT / 'ground_truth_converted_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

for n in range(1, 11):
    old = json.loads(resolve_old_gt_path(n).read_text(encoding='utf-8'))
    new = json.loads(resolve_new_gt_path(n).read_text(encoding='utf-8'))
    converted = apply_new_annotation_rules(old)

    out_path = OUT_DIR / f'sample{n}.json'
    out_path.write_text(json.dumps(converted, indent=2, ensure_ascii=False), encoding='utf-8')
    match = without_version(converted) == without_version(new)
    print(f"sample{n:<3} content match: {str(match):<6} ->  {out_path.name}")

print(f"\nSaved 10 files under: {OUT_DIR}")

sample1   content match: True   ->  sample1.json
sample2   content match: False  ->  sample2.json
sample3   content match: True   ->  sample3.json
sample4   content match: True   ->  sample4.json
sample5   content match: True   ->  sample5.json
sample6   content match: True   ->  sample6.json
sample7   content match: True   ->  sample7.json
sample8   content match: True   ->  sample8.json
sample9   content match: True   ->  sample9.json
sample10  content match: True   ->  sample10.json

Saved 10 files under: C:\Users\Nahid\dmpbridge\data\output\ground_truth_converted_test


## Notes

- **The sub-question merge is no longer an open problem.** Earlier revisions of this
  notebook recorded sample 5 as needing several sub-questions merged into one with their
  answers concatenated, and treated that as underivable from the JSON. Under the current
  table and reference files, sample 5 matches. No merging is required.
- Rows that never fire on this corpus are exercised only by
  `tests/test_annotation_rules.py`.
- Any change to `Rules.xlsx` changes how Path B scores. Re-run this notebook and the test
  suite after editing the sheet.